<a href="https://colab.research.google.com/github/PauletteG9/explainable_oscc_classification/blob/main/medkan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch

In [ ]:
%%writefile medkan_model.py
"""
MedKAN-Tiny: a scaled-down, from-scratch implementation of MedKAN
(Yang et al., 2025, arXiv:2502.18416) for OSCC H&E histopathology classification.

Architecture, translated directly from the paper's equations (Section 2):
  - Stem + patch embedding (standard conv, for initial downsampling)
  - LIK module (Local Information KAN):
        LGCK block : grouped KAN convolution + residual
                     F_LGCK = Concat_i[ ConvKAN(F_i) ] + F_patch
                     ConvKAN(X) = sum_k phi_k(X), phi_k = learnable RBF-based KAN transform
        SFFN block : standard MobileNetV2-style inverted residual (NOT KAN --
                     the paper itself specifies this block as plain conv)
  - GIK module (Global Information KAN):
        replaces multi-head self-attention with KAN layers applied to the
        flattened spatial sequence. Implemented here as an MLP-Mixer-style
        block (token-mix + channel-mix) with ordinary Linear layers replaced
        by RBF-based KANLinear layers, since the paper specifies KAN layers
        operating on the flattened sequence but does not give exact code.
  - RBF activation (not B-spline) throughout, matching the paper's Section 2.3
    choice, made for GPU parallelization.

Sizing: this is deliberately smaller than the paper's smallest published
variant (MedKAN-S, 11.5M params, trained on datasets of 780-236,386 images).
MedKAN-Tiny targets ~1-2M parameters, appropriate for a ~1,200-image OSCC
dataset with conservative augmentation. Treat this as an exploratory
secondary comparison, not a reproduction of the paper's own results, which
were obtained on much larger datasets with no public code or pretrained
weights available to validate against.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# RBF-based KAN primitives (Section 2.3: RBF replaces B-spline for parallelism)
# ---------------------------------------------------------------------------

class RBFBasis(nn.Module):
    """
    Expands each scalar input into K radial-basis-function responses:
        RBF_k(x) = exp( -(x - c_k)^2 / (2 * sigma^2) )
    Centers c_k are fixed, evenly spaced over an expected input range
    (inputs are normalized/BN'd beforehand, so a fixed range is reasonable).
    """
    def __init__(self, num_centers: int = 5, value_range: tuple = (-2.0, 2.0)):
        super().__init__()
        centers = torch.linspace(value_range[0], value_range[1], num_centers)
        self.register_buffer("centers", centers)  # (K,)
        span = (value_range[1] - value_range[0]) / max(num_centers - 1, 1)
        self.sigma = span * 0.75  # fixed width, tied to center spacing

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (..., C) -> (..., C, K)
        x = x.unsqueeze(-1)
        c = self.centers.view(*([1] * (x.dim() - 1)), -1)
        return torch.exp(-((x - c) ** 2) / (2 * self.sigma ** 2))


class RBFKANConv2d(nn.Module):
    """
    ConvKAN(X) = sum_k phi_k(X), phi_k learnable KAN transform.
    Implemented as: RBF-expand each input channel into K basis maps,
    stack along the channel dimension, then combine with a standard
    (learnable-weight) convolution. This makes the k-th basis's
    contribution phi_k(X) = W_k * RBF_k(X), i.e. a linear combination of
    K nonlinear basis functions per input channel -- exactly the ConvKAN
    definition in the paper, implemented so it runs as a single grouped
    conv rather than K separate convolutions.
    """
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, groups=1,
                 num_centers=5, value_range=(-2.0, 2.0)):
        super().__init__()
        assert in_ch % groups == 0 and out_ch % groups == 0
        self.groups = groups
        self.num_centers = num_centers
        self.rbf = RBFBasis(num_centers, value_range)
        padding = kernel_size // 2
        # After RBF expansion, channel count becomes in_ch * num_centers.
        # A grouped conv over the expanded channels realizes sum_k W_k * RBF_k(X).
        self.combine = nn.Conv2d(
            in_ch * num_centers, out_ch, kernel_size=kernel_size,
            stride=stride, padding=padding, groups=groups, bias=True,
        )
        self.norm = nn.BatchNorm2d(in_ch)  # keeps inputs within RBF's effective range

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        x = self.norm(x)
        x = torch.tanh(x)  # bound activations into RBF centers' range
        x_perm = x.permute(0, 2, 3, 1)                 # (B,H,W,C)
        basis = self.rbf(x_perm)                        # (B,H,W,C,K)
        basis = basis.permute(0, 3, 4, 1, 2)             # (B,C,K,H,W)
        basis = basis.reshape(B, C * self.num_centers, H, W)
        return self.combine(basis)


class RBFKANLinear(nn.Module):
    """Linear analogue of RBFKANConv2d, for the GIK (global) module."""
    def __init__(self, in_f, out_f, num_centers=5, value_range=(-2.0, 2.0)):
        super().__init__()
        self.num_centers = num_centers
        self.rbf = RBFBasis(num_centers, value_range)
        self.norm = nn.LayerNorm(in_f)
        self.combine = nn.Linear(in_f * num_centers, out_f)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (..., in_f)
        x = self.norm(x)
        x = torch.tanh(x)
        basis = self.rbf(x)                              # (..., in_f, K)
        basis = basis.flatten(-2, -1)                     # (..., in_f*K)
        return self.combine(basis)


# ---------------------------------------------------------------------------
# LIK module: LGCK block (grouped KAN conv) + SFFN block (plain conv, per paper)
# ---------------------------------------------------------------------------

class LGCKBlock(nn.Module):
    """F_LGCK = ConvKAN_grouped(F_patch) + F_patch"""
    def __init__(self, dim, groups=4, num_centers=5):
        super().__init__()
        self.kanconv = RBFKANConv2d(dim, dim, kernel_size=3, stride=1,
                                     groups=groups, num_centers=num_centers)

    def forward(self, x):
        return self.kanconv(x) + x


class SFFNBlock(nn.Module):
    """
    Standard MobileNetV2-style inverted residual, exactly as specified in
    the paper (Section 2.1.2): Conv1x1 -> DWConv3x3 -> Conv1x1 + residual.
    Deliberately NOT KAN-based -- this matches the paper's own design.
    """
    def __init__(self, dim, expand_ratio=2):
        super().__init__()
        hidden = dim * expand_ratio
        self.block = nn.Sequential(
            nn.Conv2d(dim, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden),
            nn.GELU(),
            nn.Conv2d(hidden, dim, 1, bias=False),
            nn.BatchNorm2d(dim),
        )

    def forward(self, x):
        return x + self.block(x)


class LIKModule(nn.Module):
    """LIK = LGCK block followed by SFFN block."""
    def __init__(self, dim, groups=4, num_centers=5, expand_ratio=2):
        super().__init__()
        self.lgck = LGCKBlock(dim, groups=groups, num_centers=num_centers)
        self.sffn = SFFNBlock(dim, expand_ratio=expand_ratio)

    def forward(self, x):
        x = self.lgck(x)
        x = self.sffn(x)
        return x


# ---------------------------------------------------------------------------
# GIK module: KAN layers replacing self-attention on the flattened sequence
# (implemented as an MLP-Mixer-style token-mix + channel-mix block, with
#  ordinary Linear layers replaced by RBFKANLinear)
# ---------------------------------------------------------------------------

class GIKModule(nn.Module):
    def __init__(self, dim, num_tokens, num_centers=5, token_hidden_ratio=1.0,
                 channel_hidden_ratio=2.0):
        super().__init__()
        token_hidden = max(int(num_tokens * token_hidden_ratio), 8)
        channel_hidden = int(dim * channel_hidden_ratio)

        self.token_norm = nn.LayerNorm(dim)
        self.token_kan1 = RBFKANLinear(num_tokens, token_hidden, num_centers)
        self.token_kan2 = RBFKANLinear(token_hidden, num_tokens, num_centers)

        self.channel_norm = nn.LayerNorm(dim)
        self.channel_kan1 = RBFKANLinear(dim, channel_hidden, num_centers)
        self.channel_kan2 = RBFKANLinear(channel_hidden, dim, num_centers)

    def forward(self, x):
        # x: (B, N, D)  where N = flattened spatial tokens, D = channel dim
        # --- token-mixing (across spatial positions; replaces self-attention) ---
        y = self.token_norm(x).transpose(1, 2)         # (B, D, N)
        y = self.token_kan1(y)
        y = self.token_kan2(y)
        x = x + y.transpose(1, 2)

        # --- channel-mixing (per-token nonlinear transform) ---
        y = self.channel_norm(x)
        y = self.channel_kan1(y)
        y = self.channel_kan2(y)
        x = x + y
        return x


# ---------------------------------------------------------------------------
# Stem, patch embedding, and full MedKAN-Tiny model
# ---------------------------------------------------------------------------

class ConvStem(nn.Module):
    """Basic convolutional module for initial feature extraction + downsampling."""
    def __init__(self, in_ch=3, out_ch=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)


class PatchEmbed(nn.Module):
    """Refines spatial/channel dims before the LIK/GIK stack."""
    def __init__(self, in_ch, out_ch, stride=2):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1)
        self.norm = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return F.gelu(self.norm(self.proj(x)))


class DownsampleStage(nn.Module):
    """Patch-merging style downsample between stages."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.norm = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return F.gelu(self.norm(self.proj(x)))


class MedKANTiny(nn.Module):
    """
    A scaled-down MedKAN for small histopathology datasets.

    Config (deliberately small; MedKAN-S in the original paper is 11.5M
    params trained on datasets of 780-236,386 images -- this targets
    roughly 1-2M params, appropriate for ~1,200 OSCC images):

        224x224 input
          -> stem (stride 2)                          -> 112x112, dims[0]
          -> patch embed (stride 2)                    -> 56x56,  dims[0]
          -> Stage 1: [LIK] x depths[0]                 (56x56,  dims[0])
          -> downsample                                 -> 28x28, dims[1]
          -> Stage 2: [LIK] x depths[1]                 (28x28,  dims[1])
          -> downsample                                 -> 14x14, dims[2]
          -> Stage 3: [LIK x1, GIK x1] x depths[2]       (14x14,  dims[2])
          -> global average pool -> classification head
    """
    def __init__(self, num_classes=2, in_ch=3, dims=(32, 64, 128),
                 depths=(1, 1, 1), groups=4, num_centers=5, img_size=224):
        super().__init__()
        self.stem = ConvStem(in_ch, dims[0])
        self.patch_embed = PatchEmbed(dims[0], dims[0], stride=2)  # -> /4 total

        self.stage1 = nn.Sequential(*[
            LIKModule(dims[0], groups=groups, num_centers=num_centers)
            for _ in range(depths[0])
        ])
        self.down1 = DownsampleStage(dims[0], dims[1])              # -> /8

        self.stage2 = nn.Sequential(*[
            LIKModule(dims[1], groups=groups, num_centers=num_centers)
            for _ in range(depths[1])
        ])
        self.down2 = DownsampleStage(dims[1], dims[2])               # -> /16

        stage3_res = img_size // 16
        num_tokens = stage3_res * stage3_res
        stage3_blocks = []
        for _ in range(depths[2]):
            stage3_blocks.append(("lik", LIKModule(dims[2], groups=groups, num_centers=num_centers)))
        self.stage3_lik = nn.ModuleList([b for _, b in stage3_blocks])
        self.gik = GIKModule(dims[2], num_tokens=num_tokens, num_centers=num_centers)

        self.norm = nn.LayerNorm(dims[2])
        self.head = nn.Linear(dims[2], num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.patch_embed(x)

        x = self.stage1(x)
        x = self.down1(x)

        x = self.stage2(x)
        x = self.down2(x)

        for blk in self.stage3_lik:
            x = blk(x)

        B, C, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)   # (B, N, C)
        tokens = self.gik(tokens)
        tokens = self.norm(tokens)
        pooled = tokens.mean(dim=1)              # global average pool over tokens
        return self.head(pooled)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    head_params = sum(p.numel() for p in model.head.parameters())
    return total, trainable, head_params


if __name__ == "__main__":
    model = MedKANTiny(num_classes=2, dims=(32, 64, 128), depths=(1, 1, 1))
    x = torch.randn(2, 3, 224, 224)
    out = model(x)
    total, trainable, head = count_parameters(model)
    print("Output shape:", out.shape)
    print(f"Total params:     {total:,}")
    print(f"Trainable params: {trainable:,}")
    print(f"Head params:      {head:,}")

Writing medkan_model.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
"""
MedKAN-Tiny for OSCC Histopathology Classification
====================================================

A from-scratch, scaled-down implementation of MedKAN (Yang et al., 2025,
arXiv:2502.18416) for binary OSCC classification (Normal vs OSCC), positioned
as an EXPLORATORY SECONDARY COMPARISON alongside your primary Swin-Tiny +
MLP/KAN-head framework -- not a replacement for it.

IMPORTANT CONTEXT (read before running):
  - MedKAN has no public code or pretrained weights. This is a from-scratch
    reimplementation built directly from the paper's equations (Section 2).
  - The original paper trains on MedMNIST datasets ranging from 780 to
    236,386 images and validates only MedKAN-S (11.5M params) and larger.
    Your OSCC dataset (~1,224 original images) is far smaller than anything
    the paper validates on, and this is a from-scratch (no transfer learning)
    architecture -- expect this to underperform your transfer-learned Swin
    baseline, likely by a meaningful margin. That is itself a legitimate,
    reportable finding for your thesis ("a from-scratch KAN-native
    architecture, even lightweight, requires more data than is available in
    OSCC-scale histopathology datasets").
  - Report this run's results as "MedKAN-Tiny (our lightweight
    reimplementation)" in your thesis, not as "MedKAN" outright, since the
    architecture, sizing and training recipe are your own adaptation, not
    the authors' code.

USAGE:
    1. Set DATA_DIR below to your dataset root, organized as:
           DATA_DIR/Normal/*.png
           DATA_DIR/OSCC/*.png
    2. Adjust MODEL_SIZE ("tiny" ~1.1M params, "small" ~2.8M params) if desired.
    3. Run: python medkan_oscc_train.py
"""

import os
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)
from medkan_model import MedKANTiny, count_parameters

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

SEED = 42
DATA_DIR = r"/content/drive/MyDrive/archive"
IMG_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

MODEL_SIZE = "tiny"   # "tiny" (~1.1M params) or "small" (~2.8M params)
MODEL_CONFIGS = {
    "tiny":  dict(dims=(32, 64, 128), depths=(1, 1, 1)),
    "small": dict(dims=(48, 96, 192), depths=(2, 2, 2)),
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ---------------------------------------------------------------------------
# Data: same conservative histopathology augmentation policy as your
# Swin+MLP/KAN pipeline, kept identical here for a fair qualitative comparison
# of architectures (not a controlled ablation -- MedKAN-Tiny is a different
# backbone entirely, so this is about consistent preprocessing, not identical-
# conditions science the way your MLP-vs-KAN head comparison is).
# ---------------------------------------------------------------------------

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def build_dataloaders():
    full_dataset = datasets.ImageFolder(DATA_DIR)
    n = len(full_dataset)
    n_test = int(n * TEST_SPLIT)
    n_val = int(n * VAL_SPLIT)
    n_train = n - n_val - n_test

    gen = torch.Generator().manual_seed(SEED)
    train_idx, val_idx, test_idx = random_split(range(n), [n_train, n_val, n_test], generator=gen)

    def subset_with_transform(indices, transform):
        ds = datasets.ImageFolder(DATA_DIR, transform=transform)
        return torch.utils.data.Subset(ds, indices)

    train_ds = subset_with_transform(train_idx.indices, train_transform)
    val_ds = subset_with_transform(val_idx.indices, eval_transform)
    test_ds = subset_with_transform(test_idx.indices, eval_transform)

    print(f"Classes: {full_dataset.classes}  (index 0={full_dataset.classes[0]}, "
          f"1={full_dataset.classes[1]})")
    print(f"Train/Val/Test sizes: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    return train_loader, val_loader, test_loader, full_dataset.classes

In [ ]:
# ---------------------------------------------------------------------------
# Train / evaluate
# ---------------------------------------------------------------------------

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total_n += images.size(0)

    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_full(model, loader, class_names):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for images, labels in loader:
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)[:, 1]  # P(OSCC), assumes class 1 = OSCC
        preds = logits.argmax(1).cpu().numpy()

        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metrics = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "sensitivity_recall": recall_score(all_labels, all_preds),
        "specificity": specificity,
        "precision": precision_score(all_labels, all_preds),
        "f1_score": f1_score(all_labels, all_preds),
        "roc_auc": roc_auc_score(all_labels, all_probs),
        "pr_auc": average_precision_score(all_labels, all_probs),
        "confusion_matrix": cm,
        "false_positives": int(fp),
        "false_negatives": int(fn),
    }
    return metrics


def print_metrics(title, metrics, model):
    total, trainable, head = count_parameters(model)
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")
    print(f"Accuracy:            {metrics['accuracy']:.4f}")
    print(f"Sensitivity (Recall):{metrics['sensitivity_recall']:.4f}")
    print(f"Specificity:         {metrics['specificity']:.4f}")
    print(f"Precision:           {metrics['precision']:.4f}")
    print(f"F1-score:            {metrics['f1_score']:.4f}")
    print(f"ROC-AUC:             {metrics['roc_auc']:.4f}")
    print(f"PR-AUC:              {metrics['pr_auc']:.4f}")
    print(f"Confusion matrix:\n{metrics['confusion_matrix']}")
    print(f"False positives:     {metrics['false_positives']}")
    print(f"False negatives:     {metrics['false_negatives']}")
    print(f"Total parameters:    {total:,}")
    print(f"Trainable parameters:{trainable:,}")
    print(f"Head parameters:     {head:,}")


In [ ]:
def main():
    set_seed(SEED)
    train_loader, val_loader, test_loader, class_names = build_dataloaders()

    cfg = MODEL_CONFIGS[MODEL_SIZE]
    model = MedKANTiny(num_classes=2, img_size=IMG_SIZE, **cfg).to(DEVICE)
    total, trainable, head = count_parameters(model)
    print(f"\nMedKAN-{MODEL_SIZE} | total params: {total:,} | trainable: {trainable:,} | head: {head:,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step()

        print(f"Epoch {epoch:3d}/{MAX_EPOCHS} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch} "
                      f"(no val_loss improvement for {EARLY_STOPPING_PATIENCE} epochs).")
                break

    model.load_state_dict(best_state)

    val_metrics = evaluate_full(model, val_loader, class_names)
    print_metrics(f"VALIDATION RESULTS -- MedKAN-{MODEL_SIZE}", val_metrics, model)

    test_metrics = evaluate_full(model, test_loader, class_names)
    print_metrics(f"TEST RESULTS -- MedKAN-{MODEL_SIZE}", test_metrics, model)

    torch.save(best_state, f"medkan_{MODEL_SIZE}_best.pt")
    print(f"\nSaved best checkpoint to medkan_{MODEL_SIZE}_best.pt")


if __name__ == "__main__":
    main()

Classes: ['test', 'train', 'val']  (index 0=test, 1=train)
Train/Val/Test sizes: 3636/778/778

MedKAN-tiny | total params: 1,111,130 | trainable: 1,111,130 | head: 258


KeyboardInterrupt: 

In [ ]:
import kagglehub
path = kagglehub.dataset_download("ashenafifasilkebede/dataset")

100%|█████████████████████████████████████████████████████████████████████████████| 2.93G/2.93G [03:08<00:00, 16.8MB/s]


Extracting files...


In [ ]:
%%writefile medkan_model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class RBFBasis(nn.Module):
    def __init__(self, num_centers: int = 5, value_range: tuple = (-2.0, 2.0)):
        super().__init__()
        centers = torch.linspace(value_range[0], value_range[1], num_centers)
        self.register_buffer("centers", centers)
        span = (value_range[1] - value_range[0]) / max(num_centers - 1, 1)
        self.sigma = float(span * 0.75)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(-1)
        c = self.centers.view(*([1] * (x.dim() - 1)), -1)
        # 1e-7 prevents gradient zero-division
        return torch.exp(-((x - c) ** 2) / (2 * (self.sigma ** 2) + 1e-7))

class RBFKANConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, groups=1,
                 num_centers=5, value_range=(-2.0, 2.0)):
        super().__init__()
        assert in_ch % groups == 0 and out_ch % groups == 0
        self.groups = groups
        self.num_centers = num_centers
        self.rbf = RBFBasis(num_centers, value_range)
        padding = kernel_size // 2

        self.combine = nn.Conv2d(
            in_ch * num_centers, out_ch, kernel_size=kernel_size,
            stride=stride, padding=padding, groups=groups, bias=True,
        )
        self.norm = nn.BatchNorm2d(in_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        x = self.norm(x)
        x = torch.clamp(x, -2.0, 2.0)
        x_perm = x.permute(0, 2, 3, 1)
        basis = self.rbf(x_perm)
        basis = basis.permute(0, 3, 4, 1, 2)
        basis = basis.reshape(B, C * self.num_centers, H, W)
        return self.combine(basis)

class RBFKANLinear(nn.Module):
    def __init__(self, in_f, out_f, num_centers=5, value_range=(-2.0, 2.0)):
        super().__init__()
        self.num_centers = num_centers
        self.rbf = RBFBasis(num_centers, value_range)
        self.norm = nn.LayerNorm(in_f)
        self.combine = nn.Linear(in_f * num_centers, out_f)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.norm(x)
        x = torch.clamp(x, -2.0, 2.0)
        basis = self.rbf(x)
        basis = basis.flatten(-2, -1)
        return self.combine(basis)

class LGCKBlock(nn.Module):
    def __init__(self, dim, groups=4, num_centers=5):
        super().__init__()
        self.kanconv = RBFKANConv2d(dim, dim, kernel_size=3, stride=1,
                                     groups=groups, num_centers=num_centers)

    def forward(self, x):
        return self.kanconv(x) + x

class SFFNBlock(nn.Module):
    def __init__(self, dim, expand_ratio=2):
        super().__init__()
        hidden = dim * expand_ratio
        self.block = nn.Sequential(
            nn.Conv2d(dim, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden),
            nn.GELU(),
            nn.Conv2d(hidden, dim, 1, bias=False),
            nn.BatchNorm2d(dim),
        )

    def forward(self, x):
        return x + self.block(x)

class LIKModule(nn.Module):
    def __init__(self, dim, groups=4, num_centers=5, expand_ratio=2):
        super().__init__()
        self.lgck = LGCKBlock(dim, groups=groups, num_centers=num_centers)
        self.sffn = SFFNBlock(dim, expand_ratio=expand_ratio)

    def forward(self, x):
        return self.sffn(self.lgck(x))

class GIKModule(nn.Module):
    def __init__(self, dim, num_tokens, num_centers=5, token_hidden_ratio=1.0,
                 channel_hidden_ratio=2.0):
        super().__init__()
        token_hidden = max(int(num_tokens * token_hidden_ratio), 8)
        channel_hidden = int(dim * channel_hidden_ratio)

        self.token_norm = nn.LayerNorm(dim)
        self.token_kan1 = RBFKANLinear(num_tokens, token_hidden, num_centers)
        self.token_kan2 = RBFKANLinear(token_hidden, num_tokens, num_centers)

        self.channel_norm = nn.LayerNorm(dim)
        self.channel_kan1 = RBFKANLinear(dim, channel_hidden, num_centers)
        self.channel_kan2 = RBFKANLinear(channel_hidden, dim, num_centers)

    def forward(self, x):
        y = self.token_norm(x).transpose(1, 2)
        y = self.token_kan1(y)
        y = self.token_kan2(y)
        x = x + y.transpose(1, 2)

        y = self.channel_norm(x)
        y = self.channel_kan1(y)
        y = self.channel_kan2(y)
        return x + y

class ConvStem(nn.Module):
    def __init__(self, in_ch=3, out_ch=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)

class PatchEmbed(nn.Module):
    def __init__(self, in_ch, out_ch, stride=2):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1)
        self.norm = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return F.gelu(self.norm(self.proj(x)))

class DownsampleStage(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.norm = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return F.gelu(self.norm(self.proj(x)))

class MedKANTiny(nn.Module):
    def __init__(self, num_classes=2, in_ch=3, dims=(32, 64, 128),
                 depths=(1, 1, 1), groups=4, num_centers=5, img_size=224):
        super().__init__()
        self.stem = ConvStem(in_ch, dims[0])
        self.patch_embed = PatchEmbed(dims[0], dims[0], stride=2)

        self.stage1 = nn.Sequential(*[
            LIKModule(dims[0], groups=groups, num_centers=num_centers)
            for _ in range(depths[0])
        ])
        self.down1 = DownsampleStage(dims[0], dims[1])

        self.stage2 = nn.Sequential(*[
            LIKModule(dims[1], groups=groups, num_centers=num_centers)
            for _ in range(depths[1])
        ])
        self.down2 = DownsampleStage(dims[1], dims[2])

        stage3_res = img_size // 16
        num_tokens = stage3_res * stage3_res
        self.stage3_lik = nn.ModuleList([
            LIKModule(dims[2], groups=groups, num_centers=num_centers)
            for _ in range(depths[2])
        ])
        self.gik = GIKModule(dims[2], num_tokens=num_tokens, num_centers=num_centers)

        self.norm = nn.LayerNorm(dims[2])
        self.head = nn.Linear(dims[2], num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.patch_embed(x)
        x = self.stage1(x)
        x = self.down1(x)
        x = self.stage2(x)
        x = self.down2(x)

        for blk in self.stage3_lik:
            x = blk(x)

        tokens = x.flatten(2).transpose(1, 2)
        tokens = self.gik(tokens)
        tokens = self.norm(tokens)
        pooled = tokens.mean(dim=1)
        return self.head(pooled)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    head_params = sum(p.numel() for p in model.head.parameters())
    return total, trainable, head_params

Writing medkan_model.py


In [ ]:
import os
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)
from medkan_model import MedKANTiny, count_parameters

SEED = 42
DATA_DIR = path
IMG_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

MODEL_SIZE = "tiny"
MODEL_CONFIGS = {
    "tiny":  dict(dims=(32, 64, 128), depths=(1, 1, 1)),
    "small": dict(dims=(48, 96, 192), depths=(2, 2, 2)),
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def build_dataloaders():
    train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
    val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"),   transform=eval_transform)
    test_ds  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"),  transform=eval_transform)

    class_names = train_ds.classes
    assert train_ds.classes == val_ds.classes == test_ds.classes, \
        f"Class mismatch across splits: {train_ds.classes}, {val_ds.classes}, {test_ds.classes}"

    print(f"Classes: {class_names}  (index 0={class_names[0]}, 1={class_names[1]})")
    print(f"Train/Val/Test sizes: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    return train_loader, val_loader, test_loader, class_names
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).long()  # Explicit long type

            if is_train:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                # Essential: stops KAN/RBF gradient explosions that crash CUDA
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total_n += images.size(0)

    return total_loss / total_n, total_correct / total_n

@torch.no_grad()
def evaluate_full(model, loader, class_names):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for images, labels in loader:
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(1).cpu().numpy()

        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metrics = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "sensitivity_recall": recall_score(all_labels, all_preds),
        "specificity": specificity,
        "precision": precision_score(all_labels, all_preds),
        "f1_score": f1_score(all_labels, all_preds),
        "roc_auc": roc_auc_score(all_labels, all_probs),
        "pr_auc": average_precision_score(all_labels, all_probs),
        "confusion_matrix": cm,
        "false_positives": int(fp),
        "false_negatives": int(fn),
    }
    return metrics

def print_metrics(title, metrics, model):
    total, trainable, head = count_parameters(model)
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")
    print(f"Accuracy:            {metrics['accuracy']:.4f}")
    print(f"Sensitivity (Recall):{metrics['sensitivity_recall']:.4f}")
    print(f"Specificity:         {metrics['specificity']:.4f}")
    print(f"Precision:           {metrics['precision']:.4f}")
    print(f"F1-score:            {metrics['f1_score']:.4f}")
    print(f"ROC-AUC:             {metrics['roc_auc']:.4f}")
    print(f"PR-AUC:              {metrics['pr_auc']:.4f}")
    print(f"Confusion matrix:\n{metrics['confusion_matrix']}")
    print(f"False positives:     {metrics['false_positives']}")
    print(f"False negatives:     {metrics['false_negatives']}")
    print(f"Total parameters:    {total:,}")
    print(f"Trainable parameters:{trainable:,}")
    print(f"Head parameters:     {head:,}")

def main():
    set_seed(SEED)
    train_loader, val_loader, test_loader, class_names = build_dataloaders()

    cfg = MODEL_CONFIGS[MODEL_SIZE]
    model = MedKANTiny(num_classes=2, img_size=IMG_SIZE, **cfg).to(DEVICE)
    total, trainable, head = count_parameters(model)
    print(f"\nMedKAN-{MODEL_SIZE} | total params: {total:,} | trainable: {trainable:,} | head: {head:,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step()

        print(f"Epoch {epoch:3d}/{MAX_EPOCHS} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch} (no val_loss improvement for {EARLY_STOPPING_PATIENCE} epochs).")
                break

    model.load_state_dict(best_state)

    val_metrics = evaluate_full(model, val_loader, class_names)
    print_metrics(f"VALIDATION RESULTS -- MedKAN-{MODEL_SIZE}", val_metrics, model)

    test_metrics = evaluate_full(model, test_loader, class_names)
    print_metrics(f"TEST RESULTS -- MedKAN-{MODEL_SIZE}", test_metrics, model)

    torch.save(best_state, f"medkan_{MODEL_SIZE}_best.pt")
    print(f"\nSaved best checkpoint to medkan_{MODEL_SIZE}_best.pt")

if __name__ == "__main__":
    main()

Classes: ['Normal', 'OSCC']  (index 0=Normal, 1=OSCC)
Train/Val/Test sizes: 4946/120/126

MedKAN-tiny | total params: 1,111,130 | trainable: 1,111,130 | head: 258
Epoch   1/60 | train_loss=0.5932 train_acc=0.6896 | val_loss=0.4624 val_acc=0.7833
Epoch   2/60 | train_loss=0.4919 train_acc=0.7671 | val_loss=0.4766 val_acc=0.8000
Epoch   3/60 | train_loss=0.4763 train_acc=0.7744 | val_loss=0.4801 val_acc=0.8000
Epoch   4/60 | train_loss=0.4338 train_acc=0.8055 | val_loss=0.7429 val_acc=0.6750
Epoch   5/60 | train_loss=0.3851 train_acc=0.8245 | val_loss=0.4996 val_acc=0.8250
Epoch   6/60 | train_loss=0.3584 train_acc=0.8407 | val_loss=0.5038 val_acc=0.8167
Epoch   7/60 | train_loss=0.3398 train_acc=0.8520 | val_loss=0.4049 val_acc=0.8500
Epoch   8/60 | train_loss=0.3220 train_acc=0.8617 | val_loss=0.5490 val_acc=0.8000
Epoch   9/60 | train_loss=0.2928 train_acc=0.8698 | val_loss=0.4530 val_acc=0.8500
Epoch  10/60 | train_loss=0.2873 train_acc=0.8797 | val_loss=0.6556 val_acc=0.7500
Epoch  